# 01 — Generate the acquisition-matched synthetic VERDICT dataset

This notebook generates the clean synthetic dataset for the nine candidate VERDICT-related compartment models used in this study.

The compartment signal models are implemented using **microTorch**, with the brain-tumour-specific model combinations, parameter ranges, sampling rules and custom variable-diffusivity Sphere defined here.

## Acquisition

Synthetic signals are generated using the supplied `acquisition/NV.scheme`. The full 153-measurement scheme is used for simulation, after which the 22 reference measurements are removed to produce the ordered 131-element input used by the MLP.

## Outputs

This notebook generates:

- 200,000 synthetic signals per model (1,800,000 total);
- the generating-model label for each signal;
- sampled model parameters and compartment fractions;
- fixed 80/10/10 training, validation and test splits;
- acquisition metadata required for the 131-feature input.

The dataset is stored without added measurement noise. Rician noise is introduced during MLP training in `02_train_mlp.ipynb`.

In [ ]:
from pathlib import Path
import gc
import json

import numpy as np
import torch

from microtorch.model_maker import ModelMaker
from microtorch.utils.acquisition_scheme import acquisition_scheme_loader


# Repository paths
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

ACQUISITION_FILE = PROJECT_ROOT / "acquisition" / "NV.scheme"
DATA_DIR = PROJECT_ROOT / "data" / "synthetic"
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not ACQUISITION_FILE.exists():
    raise FileNotFoundError(
        f"Could not find acquisition/NV.scheme at {ACQUISITION_FILE}"
    )

print("Acquisition:", ACQUISITION_FILE)
print("Output:", DATA_DIR)

## Parameter ranges and sampling rules

The synthetic signals are generated using the following parameter ranges and constraints. Unless otherwise stated, the model parameters are sampled uniformly over the ranges.

| Parameter | Range or sampling rule |
|---|---|
| Tissue Ball diffusivity | 0.01–3 µm²/ms |
| Zeppelin diffusivities | 0.01–3 µm²/ms, ordered so that D⊥ ≤ D∥ |
| Tensor diffusivities | 0.01–3 µm²/ms, ordered so that D⊥₂ ≤ D⊥₁ ≤ D∥ |
| Sphere radius | 0.1–20 µm |
| Sphere intracellular diffusivity | 0.01–3 µm²/ms |
| Vascular Ball / Stick diffusivity | 3–100 µm²/ms |
| AstroSticks diffusivity | 9–100 µm²/ms |
| Zeppelin / Tensor / Stick orientation | isotropically sampled in 3D |
| Tensor rotation ψ | 0–π |
| Two-compartment fractions | both fractions ≥ 0.01 and sum to 1 |
| Three-compartment fractions | symmetric Dirichlet sampling with all fractions ≥ 0.01 and sum to 1 |

For three-compartment models, the Dirichlet draw is mapped onto the constrained simplex so that every compartment contributes at least 0.01.

## Nine candidate model classes

| Class | Model | Extracellular–extravascular (EES) component | Vascular component | Intracellular component |
|---:|---|---|---|---|
| 0 | Ball | Ball | — | — |
| 1 | BallSphere | Ball | — | Sphere |
| 2 | ZeppelinSphere | Zeppelin | — | Sphere |
| 3 | TensorSphere | Tensor | — | Sphere |
| 4 | BallSphereStick | Ball | Stick | Sphere |
| 5 | ZeppelinBallSphere | Zeppelin | Vascular Ball | Sphere |
| 6 | ZeppelinAstroSticksSphere | Zeppelin | AstroSticks | Sphere |
| 7 | TensorBallSphere | Tensor | Vascular Ball | Sphere |
| 8 | TensorAstroSticksSphere | Tensor | AstroSticks | Sphere |

The rows above give the fixed class order used by the nine-output MLP.

In [ ]:
# Convert NV.scheme to the numeric acquisition format used by microTorch.
# Columns in NV.scheme:
# gx, gy, gz, G [T/m], Delta [s], delta [s], TE [s]

raw_scheme = np.loadtxt(
    ACQUISITION_FILE,
    skiprows=1,  # skip "VERSION: STEJSKALTANNER"
)

if raw_scheme.shape != (153, 7):
    raise ValueError(
        f"Expected 153 acquisition rows and 7 columns, found {raw_scheme.shape}."
    )

bvecs = raw_scheme[:, :3]
G = raw_scheme[:, 3]
Delta = raw_scheme[:, 4]
delta = raw_scheme[:, 5]
TE = raw_scheme[:, 6]

# Calculate b-values using the Stejskal–Tanner equation.
gamma = 2.675987e8  # rad/s/T
b_s_per_m2 = (gamma * G * delta) ** 2 * (Delta - delta / 3.0)
b_s_per_mm2 = b_s_per_m2 / 1e6

# microTorch format:
# gx, gy, gz, b [s/mm²], Delta [ms], delta [ms], TE [ms]
converted_scheme = np.column_stack(
    [
        bvecs,
        b_s_per_mm2,
        Delta * 1000,
        delta * 1000,
        TE * 1000,
    ]
)

CONVERTED_SCHEME_FILE = (
    ACQUISITION_FILE.parent / "NV_microtorch_bvals.generated.txt"
)

np.savetxt(
    CONVERTED_SCHEME_FILE,
    converted_scheme,
    fmt="%.8f",
)

grad = acquisition_scheme_loader(str(CONVERTED_SCHEME_FILE))

# Identify the 22 reference measurements and 131 DW measurements.
b0_mask = grad.bvalues < 1e-5
dw_mask = ~b0_mask
dw_indices = torch.where(dw_mask)[0]

assert int(grad.number_of_measurements) == 153
assert int(b0_mask.sum()) == 22
assert int(dw_mask.sum()) == 131

print("Total measurements:", int(grad.number_of_measurements))
print("Reference measurements:", int(b0_mask.sum()))
print("Diffusion-weighted measurements:", int(dw_mask.sum()))

### Variable-diffusivity Sphere

The Sphere compartment is implemented with both cell radius and intracellular diffusivity (`D_IC`) as free parameters.

In [ ]:
class SphereVariableD:
    """
    implemented with variable radius and intracellular diffusivity.

    Parameters
    ----------
    radius : µm
    D_IC   : µm²/ms
    """

    def __init__(self):
        self.parameter_ranges = [[0.1, 20.0], [0.01, 3.0]]
        self.parameter_names = ["radius", "D_IC"]
        self.n_parameters = 2
        self.spherical_mean = True

    def __call__(self, grad, parameters):
        device = parameters.device
        dtype = parameters.dtype

        b_values_all = grad.bvalues.to(device=device, dtype=dtype)
        delta_all = grad.delta.to(device=device, dtype=dtype)
        Delta_all = grad.Delta.to(device=device, dtype=dtype)

        n_samples = parameters.shape[0]
        n_measurements = b_values_all.shape[0]

        signal = torch.ones(
            (n_samples, n_measurements),
            device=device,
            dtype=dtype,
        )

        valid = (
            (b_values_all > 1e-5)
            & (delta_all > 0)
            & (Delta_all > delta_all / 3)
        )

        if not bool(valid.any()):
            return signal

        b_values = b_values_all[valid]
        delta = delta_all[valid]
        Delta = Delta_all[valid]

        radius = parameters[:, 0:1]
        D_IC = parameters[:, 1:2]
        # Precomputed roots.
        roots = torch.tensor(
            [
                2.081575978, 5.940369990, 9.205840145, 12.40444502,
                15.57923641, 18.74264558, 21.89969648, 25.05282528,
                28.20336100, 31.35209173, 34.49951492, 37.64596032,
                40.79165523, 43.93676147, 47.08139741, 50.22565165,
                53.36959180, 56.51327045, 59.65672900, 62.80000055,
                65.94311190, 69.08608495, 72.22893775, 75.37168540,
                78.51434055, 81.65691380, 84.79941440, 87.94185005,
                91.08422750, 94.22655255, 97.36883035,
            ],
            device=device,
            dtype=dtype,
        )

        alpha = roots / radius
        alpha2 = alpha**2
        alpha2_D = alpha2 * D_IC

        alpha = alpha.unsqueeze(1)
        alpha2 = alpha2.unsqueeze(1)
        alpha2_D = alpha2_D.unsqueeze(1)

        gamma = 267.5987

        gradient_strength = torch.sqrt(b_values) / (
            gamma * delta * torch.sqrt(Delta - delta / 3)
        )
        first_factor = -(gamma * gradient_strength) ** 2

        Delta = Delta.unsqueeze(0).unsqueeze(2)
        delta = delta.unsqueeze(0).unsqueeze(2)

        summands = (
            alpha**(-4)
            / (alpha2 * radius.unsqueeze(2) ** 2 - 2)
            * (
                2 * delta
                - (
                    2
                    + torch.exp(-alpha2_D * (Delta - delta))
                    - 2 * torch.exp(-alpha2_D * delta)
                    - 2 * torch.exp(-alpha2_D * Delta)
                    + torch.exp(-alpha2_D * (Delta + delta))
                )
                / alpha2_D
            )
        )

        signal_valid = torch.exp(
            first_factor * summands.sum(dim=2)
        )

        signal[:, valid] = signal_valid
        return signal


sphere_variable_model = SphereVariableD()

In [ ]:
def to_torch(x):
    """convert numpty array to pytorch."""
    return torch.tensor(x, dtype=torch.float32)


def sample_isotropic_microtorch_angles(n, rng):
    """sample 3D axes for microTorch theta/phi."""

    # Sample random 3D directions and normalise to unit length.
    vectors = rng.normal(size=(n, 3))
    vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)

    # Fold opposite directions onto the same unoriented axis.
    flip = vectors[:, 2] < 0
    vectors[flip] *= -1

    x = vectors[:, 0]
    y = vectors[:, 1]
    z = vectors[:, 2]

    radial = np.sqrt(x**2 + y**2)
    theta_abs = np.arctan2(radial, z)

    theta = theta_abs.copy()
    phi = np.zeros(n)

    positive_y = y >= 0
    phi[positive_y] = np.arctan2(
        y[positive_y],
        x[positive_y],
    )

    negative_y = ~positive_y
    theta[negative_y] = -theta_abs[negative_y]
    phi[negative_y] = np.arctan2(
        -y[negative_y],
        -x[negative_y],
    )

    # keep angles within the convention used by microTorch.
    theta = np.clip(theta, -np.pi / 2, np.pi / 2)
    phi = np.clip(phi, 0.0, np.pi)

    return theta, phi


def sample_ball_single_np(n, rng):
    """uniform sampling of tissue ball diffusivity."""
    D = rng.uniform(0.01, 3.0, size=(n, 1))

    return {
        "params": to_torch(D),
        "D": to_torch(D),
    }


def sample_ball_vascular_np(n, rng):
    """sample ball pseudo-diffusivity."""
    d_v = rng.uniform(3.0, 100.0, size=(n, 1))

    return {
        "params": to_torch(d_v),
        "d_v": to_torch(d_v),
    }


def sample_astrosticks_vascular_np(n, rng):
    """sample astroSticks vascular pseudo-diffusivity."""
    d_v = rng.uniform(9.0, 100.0, size=(n, 1))

    return {
        "params": to_torch(d_v),
        "d_v": to_torch(d_v),
    }


def sample_stick_vascular_np(n, rng):
    """sample stick diffusivity and orientation."""
    d_stick = rng.uniform(3.0, 100.0, size=(n, 1))
    theta, phi = sample_isotropic_microtorch_angles(n, rng)

    # match samples to the parameter order implemented in microTorch.
    parameter_names = list(stick_model.parameter_names)

    if len(parameter_names) != 3:
        raise RuntimeError(
            f"Expected three Stick parameters, found {parameter_names}."
        )

    columns = []

    for name in parameter_names:
        key = str(name).replace("_", "").replace("-", "").lower()

        if key in {
            "d",
            "dpar",
            "dparallel",
            "diffusivity",
            "paralleldiffusivity",
        }:
            columns.append(d_stick[:, 0])

        elif key == "theta":
            columns.append(theta)

        elif key == "phi":
            columns.append(phi)

        else:
            raise RuntimeError(
                f"Unrecognised Stick parameter name {name!r}."
            )

    return {
        "params": to_torch(np.column_stack(columns)),
        "d_stick": to_torch(d_stick),
        "theta": to_torch(theta[:, None]),
        "phi": to_torch(phi[:, None]),
    }


def sample_sphere_brain_np(n, rng):
    """sample sphere radius and intracellular diffusivity."""
    radius = rng.uniform(0.1, 20.0, size=(n, 1))
    D_IC = rng.uniform(0.01, 3.0, size=(n, 1))

    params = np.column_stack([
        radius[:, 0],
        D_IC[:, 0],
    ])

    return {
        "params": to_torch(params),
        "radius": to_torch(radius),
        "D_IC": to_torch(D_IC),
    }


def sample_zeppelin_brain_np(n, rng):
    """sample zeppelin diffusivities and orientation."""

    # Draw the two diffusivities and order them in order to have Dperp <= Dpar.
    d = rng.uniform(0.01, 3.0, size=(n, 2))
    d.sort(axis=1)

    Dperp = d[:, 0]
    Dpar = d[:, 1]

    # convert to microtorch representation using k = Dperp / Dpar.
    k = Dperp / Dpar

    theta, phi = sample_isotropic_microtorch_angles(n, rng)

    params = np.column_stack([
        Dpar,
        k,
        theta,
        phi,
    ])

    return {
        "params": to_torch(params),
        "Dpar": to_torch(Dpar[:, None]),
        "Dperp": to_torch(Dperp[:, None]),
        "k": to_torch(k[:, None]),
        "theta": to_torch(theta[:, None]),
        "phi": to_torch(phi[:, None]),
    }


def sample_tensor_brain_np(n, rng):
    """sample Tensor diffusivities and orientation."""

    # Draw the diffusivities and order them:
    # Dperp2 <= Dperp1 <= Dpar.
    d = rng.uniform(0.01, 3.0, size=(n, 3))
    d.sort(axis=1)

    Dperp2 = d[:, 0]
    Dperp1 = d[:, 1]
    Dpar = d[:, 2]

    # convert for microTorch.
    k1 = Dperp1 / Dpar
    k2 = Dperp2 / Dperp1

    theta, phi = sample_isotropic_microtorch_angles(n, rng)
    psi = rng.uniform(0.0, np.pi, size=n)

    params = np.column_stack([
        Dpar,
        k1,
        k2,
        theta,
        phi,
        psi,
    ])

    return {
        "params": to_torch(params),
        "Dpar": to_torch(Dpar[:, None]),
        "Dperp1": to_torch(Dperp1[:, None]),
        "Dperp2": to_torch(Dperp2[:, None]),
        "k1": to_torch(k1[:, None]),
        "k2": to_torch(k2[:, None]),
        "theta": to_torch(theta[:, None]),
        "phi": to_torch(phi[:, None]),
        "psi": to_torch(psi[:, None]),
    }


def sample_fractions_2_np(
    n,
    rng,
    min_ees=0.01,
    min_ic=0.01,
):
    """sample the two-compartment fractions making sure that the min values are satisfied."""
    if min_ees + min_ic >= 1.0:
        raise ValueError("min_ees + min_ic must be < 1.")

    f_ees = rng.uniform(
        min_ees,
        1.0 - min_ic,
        size=(n, 1),
    )

    f_ic = 1.0 - f_ees

    return {
        "f_ees": to_torch(f_ees),
        "f_ic": to_torch(f_ic),
    }


def sample_fractions_3_np(
    n,
    rng,
    min_ees=0.01,
    min_vasc=0.01,
    min_ic=0.01,
):
    """Ssample three-compartment fractions from a constrained simplex."""

    minimums = np.array(
        [min_ees, min_vasc, min_ic],
        dtype=float,
    )

    if minimums.sum() >= 1.0:
        raise ValueError("Fraction minima must sum to < 1.")

    remaining = 1.0 - minimums.sum()

    # symmetric Dirichlet sampling *to make sure taht no compartment has some bias.
    raw = rng.dirichlet(
        alpha=np.ones(3),
        size=n,
    )

    # map the samples onto the simplex wit min fractions.
    fractions = minimums[None, :] + remaining * raw

    return {
        "fractions": to_torch(fractions),
        "f_ees": to_torch(fractions[:, 0:1]),
        "f_vasc": to_torch(fractions[:, 1:2]),
        "f_ic": to_torch(fractions[:, 2:3]),
    }


def force_b0_to_one(signal, grad, threshold=1e-5):
    """set reference measurements to normalise the signal value to be S0 = 1."""
    signal = signal.clone()

    mask = grad.bvalues < threshold
    signal[:, mask] = 1.0

    return signal


def keep_dw_only(signal, dw_mask):
    """uses only the 131 measurements required for the MLP."""
    signal_dw = signal[:, dw_mask].contiguous()

    assert signal_dw.shape[1] == 131

    return signal_dw

## Generation of the 9 models

In [ ]:
DATA_GENERATION_SEED = 42
N_PER_MODEL = 200_000

rng = np.random.default_rng(DATA_GENERATION_SEED)
torch.manual_seed(DATA_GENERATION_SEED)

MODEL_ORDER = [
    "Ball",
    "BallSphere",
    "ZeppelinSphere",
    "TensorSphere",
    "BallSphereStick",
    "ZeppelinBallSphere",
    "ZeppelinAstrosticksSphere",
    "TensorBallSphere",
    "TensorAstrosticksSphere",
]

LABEL_NAMES = {
    i: name
    for i, name in enumerate(MODEL_ORDER)
}

print("Seed:", DATA_GENERATION_SEED)
print("Samples/model:", f"{N_PER_MODEL:,}")
print("Classes:", len(MODEL_ORDER))

In [ ]:
# Generated with seed 42.

# Ball
ball = sample_ball_single_np(N_PER_MODEL, rng)
S_ball = keep_dw_only(
    force_b0_to_one(
        ball_model(grad, ball["params"]),
        grad,
    ),
    dw_mask,
)


# ZeppelinSphere
zs_zep = sample_zeppelin_brain_np(N_PER_MODEL, rng)
zs_sph = sample_sphere_brain_np(N_PER_MODEL, rng)
zs_frac = sample_fractions_2_np(N_PER_MODEL, rng)

S_zs = (
    zs_frac["f_ees"] * zeppelin_model(grad, zs_zep["params"])
    + zs_frac["f_ic"] * sphere_variable_model(grad, zs_sph["params"])
)

S_zs = keep_dw_only(
    force_b0_to_one(S_zs, grad),
    dw_mask,
)
gc.collect()


# ZeppelinBallSphere
zbs_zep = sample_zeppelin_brain_np(N_PER_MODEL, rng)
zbs_ballv = sample_ball_vascular_np(N_PER_MODEL, rng)
zbs_sph = sample_sphere_brain_np(N_PER_MODEL, rng)
zbs_frac = sample_fractions_3_np(N_PER_MODEL, rng)

S_zbs = (
    zbs_frac["f_ees"] * zeppelin_model(grad, zbs_zep["params"])
    + zbs_frac["f_vasc"] * ball_model(grad, zbs_ballv["params"])
    + zbs_frac["f_ic"] * sphere_variable_model(grad, zbs_sph["params"])
)

S_zbs = keep_dw_only(
    force_b0_to_one(S_zbs, grad),
    dw_mask,
)
gc.collect()


# ZeppelinAstrosticksSphere
zas_zep = sample_zeppelin_brain_np(N_PER_MODEL, rng)
zas_ast = sample_astrosticks_vascular_np(N_PER_MODEL, rng)
zas_sph = sample_sphere_brain_np(N_PER_MODEL, rng)
zas_frac = sample_fractions_3_np(N_PER_MODEL, rng)

S_zas = (
    zas_frac["f_ees"] * zeppelin_model(grad, zas_zep["params"])
    + zas_frac["f_vasc"] * astro_model(grad, zas_ast["params"])
    + zas_frac["f_ic"] * sphere_variable_model(grad, zas_sph["params"])
)

S_zas = keep_dw_only(
    force_b0_to_one(S_zas, grad),
    dw_mask,
)
gc.collect()


# TensorAstrosticksSphere
tas_tensor = sample_tensor_brain_np(N_PER_MODEL, rng)
tas_ast = sample_astrosticks_vascular_np(N_PER_MODEL, rng)
tas_sph = sample_sphere_brain_np(N_PER_MODEL, rng)
tas_frac = sample_fractions_3_np(N_PER_MODEL, rng)

S_tas = (
    tas_frac["f_ees"] * tensor_model(grad, tas_tensor["params"])
    + tas_frac["f_vasc"] * astro_model(grad, tas_ast["params"])
    + tas_frac["f_ic"] * sphere_variable_model(grad, tas_sph["params"])
)

S_tas = keep_dw_only(
    force_b0_to_one(S_tas, grad),
    dw_mask,
)
gc.collect()


# TensorBallSphere
tbs_tensor = sample_tensor_brain_np(N_PER_MODEL, rng)
tbs_ballv = sample_ball_vascular_np(N_PER_MODEL, rng)
tbs_sph = sample_sphere_brain_np(N_PER_MODEL, rng)
tbs_frac = sample_fractions_3_np(N_PER_MODEL, rng)

S_tbs = (
    tbs_frac["f_ees"] * tensor_model(grad, tbs_tensor["params"])
    + tbs_frac["f_vasc"] * ball_model(grad, tbs_ballv["params"])
    + tbs_frac["f_ic"] * sphere_variable_model(grad, tbs_sph["params"])
)

S_tbs = keep_dw_only(
    force_b0_to_one(S_tbs, grad),
    dw_mask,
)
gc.collect()


# BallSphere
bs_ball = sample_ball_single_np(N_PER_MODEL, rng)
bs_sph = sample_sphere_brain_np(N_PER_MODEL, rng)
bs_frac = sample_fractions_2_np(N_PER_MODEL, rng)

S_bs = (
    bs_frac["f_ees"] * ball_model(grad, bs_ball["params"])
    + bs_frac["f_ic"] * sphere_variable_model(grad, bs_sph["params"])
)

S_bs = keep_dw_only(
    force_b0_to_one(S_bs, grad),
    dw_mask,
)
gc.collect()


# TensorSphere
ts_tensor = sample_tensor_brain_np(N_PER_MODEL, rng)
ts_sph = sample_sphere_brain_np(N_PER_MODEL, rng)
ts_frac = sample_fractions_2_np(N_PER_MODEL, rng)

S_ts = (
    ts_frac["f_ees"] * tensor_model(grad, ts_tensor["params"])
    + ts_frac["f_ic"] * sphere_variable_model(grad, ts_sph["params"])
)

S_ts = keep_dw_only(
    force_b0_to_one(S_ts, grad),
    dw_mask,
)
gc.collect()


# BallSphereStick
bss_ball = sample_ball_single_np(N_PER_MODEL, rng)
bss_stick = sample_stick_vascular_np(N_PER_MODEL, rng)
bss_sph = sample_sphere_brain_np(N_PER_MODEL, rng)
bss_frac = sample_fractions_3_np(N_PER_MODEL, rng)

S_bss = (
    bss_frac["f_ees"] * ball_model(grad, bss_ball["params"])
    + bss_frac["f_vasc"] * stick_model(grad, bss_stick["params"])
    + bss_frac["f_ic"] * sphere_variable_model(grad, bss_sph["params"])
)

S_bss = keep_dw_only(
    force_b0_to_one(S_bss, grad),
    dw_mask,
)
gc.collect()


print("9 clean model classes are created.")

In [ ]:
X_clean = torch.cat(
    [
        S_ball,
        S_bs,
        S_zs,
        S_ts,
        S_bss,
        S_zbs,
        S_zas,
        S_tbs,
        S_tas,
    ],
    dim=0,
)

y = torch.cat(
    [
        torch.full((N_PER_MODEL,), i, dtype=torch.long)
        for i in range(len(MODEL_ORDER))
    ],
    dim=0,
)

sample_ids = torch.arange(X_clean.shape[0])

expected_n = len(MODEL_ORDER) * N_PER_MODEL

assert tuple(X_clean.shape) == (expected_n, 131)

del S_ball, S_bs, S_zs, S_ts, S_bss, S_zbs, S_zas, S_tbs, S_tas
gc.collect()

print("X_clean:", tuple(X_clean.shape))
print("y:", tuple(y.shape))
print("Class counts:", torch.bincount(y, minlength=9).tolist())

In [ ]:
row_ranges = {
    model_name: [i * N_PER_MODEL, (i + 1) * N_PER_MODEL]
    for i, model_name in enumerate(MODEL_ORDER)
}

true_parameters = {
    "sample_ids": sample_ids,
    "row_ranges": row_ranges,
    "Ball": {
        "params": ball["params"],
        "param_names": ["D"],
        "D": ball["D"],
    },
    "BallSphere": {
        "ball": bs_ball,
        "sphere": bs_sph,
        "fractions": bs_frac,
    },
    "ZeppelinSphere": {
        "zeppelin": zs_zep,
        "sphere": zs_sph,
        "fractions": zs_frac,
    },
    "TensorSphere": {
        "tensor": ts_tensor,
        "sphere": ts_sph,
        "fractions": ts_frac,
    },
    "BallSphereStick": {
        "ball": bss_ball,
        "vascular_stick": bss_stick,
        "sphere": bss_sph,
        "fractions": bss_frac,
    },
    "ZeppelinBallSphere": {
        "zeppelin": zbs_zep,
        "vascular_ball": zbs_ballv,
        "sphere": zbs_sph,
        "fractions": zbs_frac,
    },
    "ZeppelinAstrosticksSphere": {
        "zeppelin": zas_zep,
        "astrosticks": zas_ast,
        "sphere": zas_sph,
        "fractions": zas_frac,
    },
    "TensorBallSphere": {
        "tensor": tbs_tensor,
        "vascular_ball": tbs_ballv,
        "sphere": tbs_sph,
        "fractions": tbs_frac,
    },
    "TensorAstrosticksSphere": {
        "tensor": tas_tensor,
        "astrosticks": tas_ast,
        "sphere": tas_sph,
        "fractions": tas_frac,
    },
}

metadata = {
    "dataset_name": "verdict_9class_clean_seed42",
    "microtorch_historical_commit": "b67c7a069bc28b19d3aaaa0ebb2ba7e67fde979d",
    "data_generation_seed": DATA_GENERATION_SEED,
    "n_per_model": N_PER_MODEL,
    "n_total": int(X_clean.shape[0]),
    "signal_length": int(X_clean.shape[1]),
    "original_acquisition_length": int(grad.number_of_measurements),
    "n_b0_removed": int(b0_mask.sum()),
    "n_dw_retained": int(dw_mask.sum()),
    "dw_indices": [int(i) for i in dw_indices.tolist()],
    "model_order": MODEL_ORDER,
    "label_names": LABEL_NAMES,
    "row_ranges": row_ranges,
    "acquisition_file": "acquisition/NV.scheme",
    "generated_microtorch_scheme_file": "acquisition/NV_microtorch_bvals.generated.txt",
    "fraction_minimum": 0.01,
    "ml_input_contract": {
        "input_length": 131,
        "reference_rows_removed": 22,
        "preserve_original_dw_order": True,
        "signal_normalisation": "S/S0 with reference signal fixed to 1",
    },
    "note": (
        "Clean normalised signals only. Dynamic Rician noise with SNR "
        "sampled from 30–70 is added during MLP training."
    ),
}

torch.save(X_clean, DATA_DIR / "X_clean.pt")
torch.save(y, DATA_DIR / "y_labels.pt")
torch.save(true_parameters, DATA_DIR / "true_parameters.pt")
torch.save(metadata, DATA_DIR / "metadata.pt")

print("Saved clean dataset files to:", DATA_DIR)

## Create the fixed 80/10/10 ML split

The split is created once with `random_state=42` and reused throughout training and evaluation. Each class contributes 160,000 training, 20,000 validation, and 20,000 held-out test samples.

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

y_np = y.cpu().numpy().astype(np.int64)
all_idx = np.arange(len(y_np), dtype=np.int64)

# 80% training, 20% temporary holdout.
train_idx, holdout_idx = train_test_split(
    all_idx,
    test_size=0.20,
    stratify=y_np,
    random_state=RANDOM_STATE,
)

# Split into validation 10% and test sets 10%.
val_idx, test_idx = train_test_split(
    holdout_idx,
    test_size=0.50,
    stratify=y_np[holdout_idx],
    random_state=RANDOM_STATE,
)

train_idx = np.asarray(train_idx, dtype=np.int64)
val_idx = np.asarray(val_idx, dtype=np.int64)
test_idx = np.asarray(test_idx, dtype=np.int64)


split_file = DATA_DIR / "master_split_80train_10val_10test_seed42.npz"

np.savez_compressed(
    split_file,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
    random_state=np.array([RANDOM_STATE], dtype=np.int64),
    label_names=np.array(MODEL_ORDER),
)

print("Saved split:", split_file)
print(
    "Train / validation / test:",
    len(train_idx),
    len(val_idx),
    len(test_idx),
)

### End of notebook 01

`02_train_final_mlp.ipynb` loads only the clean synthetic signals and fixed split, then adds fresh Rician \(SNR\sim U(30,70)\) noise during training.